# ATLAS Wind IRI Climatology Datasets Preprocessing

## Overview

This notebook preprocesses monthly wind climatology data from the International Research Institute for Climate and Society (IRI) Data Library:

https://iridl.ldeo.columbia.edu/

The dataset provides monthly climatological wind fields derived from the NOAA NCEP/NCAR CDAS-1 reanalysis and is available through the IRI Global Climatologies portal:

https://iridl.ldeo.columbia.edu/maproom/Global/Climatologies/Vector_Winds.html

The variables used in this workflow are:

* **U wind component** (zonal wind)
* **V wind component** (meridional wind)

The original data source can be accessed through:

https://iridl.ldeo.columbia.edu/ds:/SOURCES/.NOAA/.NCEP-NCAR/.CDAS-1/.mc9120/.Intrinsic/.PressureLevel/.u/decompress/dup/mul/SOURCES/.NOAA/.NCEP-NCAR/.CDAS-1/.mc9120/.Intrinsic/.PressureLevel/.v/decompress/dup/mul/add/sqrt//long_name/%28speed%5Bm/s%5D%29def/windspeed_anomaly_colors/DATA/0/3/6/9/12/15/18/21/24/27/30/VALUES//name/%28speed%29def/SOURCES/.NOAA/.NCEP-NCAR/.CDAS-1/.mc9120/.Intrinsic/.PressureLevel/.u//name/%28u%29def/SOURCES/.NOAA/.NCEP-NCAR/.CDAS-1/.mc9120/.Intrinsic/.PressureLevel/.v//name/%28v%29def/:ds/

## Download Instructions

To download the original data:

1. Open the IRI dataset page.
2. Select the desired variable (**u** or **v**).
3. Navigate to **Data Files**.
4. Download the dataset in **NetCDF** format.

## Project Data Availability

For the purposes of the ATLAS workflow, the required NetCDF files are already provided and do not need to be downloaded manually.

They are expected to be available in:

```text
../data/IRI/
```

## Purpose of this Notebook

This notebook:

1. Loads the monthly climatological U and V wind components.
2. Harmonizes coordinates and metadata.
3. Converts longitude coordinates to the ATLAS convention.
4. Extracts the South American domain.
5. Saves the processed datasets in a format ready for the ATLAS downscaling workflow.

## Step 1. User configuration

Edit only this section before running the notebook.

Use generic project folders rather than machine specific paths. The default structure assumes that raw files are stored under `data/raw` and outputs are written under `data/processed`.

In [8]:
from pathlib import Path

# Main project folder. Change this path according to your local machine or server.
PROJECT_DIR = Path("..")

# Input folders
RAW_DATA_DIR = PROJECT_DIR / "data" / "iri"
OUTPUT_DIR = PROJECT_DIR / "data" / "processed"

# Input files
INPUT_U_FILE = RAW_DATA_DIR / "u_monthly_clim_plev_global.nc"
INPUT_V_FILE = RAW_DATA_DIR / "v_monthly_clim_plev_global.nc"

# Output files
OUTPUT_U_FILE = OUTPUT_DIR / "u_clim" / "u_iri_processed_global.nc"
OUTPUT_V_FILE = OUTPUT_DIR / "v_clim" / "v_iri_processed_global.nc"

# Variable names inside the input NetCDF files
U_VARIABLE_NAME = "u"
V_VARIABLE_NAME = "v"

# Pressure level to extract, if a pressure dimension is present
TARGET_PRESSURE_LEVEL = 1000

# NetCDF compression level. Use 1 for faster output, 4 for a good compromise, 9 for maximum compression.
NETCDF_COMPRESSION_LEVEL = 4

# Set to True to overwrite existing output files
OVERWRITE_OUTPUTS = True

## Step 2. Import libraries and create output folders

Run this cell once at the beginning of the workflow.

In [9]:
import os
import gc
import time
import warnings

import numpy as np
import xarray as xr

try:
    import psutil
    PSUTIL_AVAILABLE = True
except ImportError:
    PSUTIL_AVAILABLE = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"Project folder: {PROJECT_DIR.resolve()}")
print(f"Raw data folder: {RAW_DATA_DIR.resolve()}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")

Configuration loaded.
Project folder: /home/python/jupyters/WMO_ATLAS/Notebooks_for_Deliverable_3
Raw data folder: /home/python/jupyters/WMO_ATLAS/Notebooks_for_Deliverable_3/data/iri
Output folder: /home/python/jupyters/WMO_ATLAS/Notebooks_for_Deliverable_3/data/processed


## Step 3. Helper functions

These functions are shared by the U and V component processing steps.

In [10]:
def print_step(message):
    """Print a timestamped workflow message."""
    timestamp = time.strftime("%H:%M:%S")
    print(f"[{timestamp}] {message}")


def print_memory_usage(label):
    """Print current Python process memory usage when psutil is available."""
    if not PSUTIL_AVAILABLE:
        print(f"{label}: memory check skipped because psutil is not installed.")
        return

    process = psutil.Process(os.getpid())
    ram_gb = process.memory_info().rss / 1024**3
    print(f"{label}: {ram_gb:.2f} GB used by the current Python process")


def check_input_file(path):
    """Check that an input file exists before processing."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path}"
            "Please update RAW_DATA_DIR or the input file names in Step 1."
        )
    print(f"Input file found: {path}")


def preview_dataset(path):
    """Open and display a small summary of a NetCDF file."""
    path = Path(path)
    check_input_file(path)
    ds = xr.open_dataset(path, decode_times=False)
    print(ds)
    return ds

## Step 4. Preprocessing function

The function below contains the scientific processing logic.

Main operations:

1. Open the source NetCDF file without decoding time immediately.
2. Decode the `T` coordinate using the `360_day` calendar when present.
3. Select pressure level `P = 1000` when a pressure dimension is available.
4. Convert the time dimension `T` into a `month` dimension from 1 to 12.
5. Rename spatial coordinates from `Y` and `X` to `latitude` and `longitude`.
6. Convert longitudes from `0 / 360` to `-180 / 180`.
7. Sort latitude and longitude consistently.
8. Save a compressed NetCDF file in EPSG:4326.

In [11]:
def preprocess_iridl_wind(
    input_path,
    output_path,
    var_name=None,
    target_pressure_level=1000,
    compression_level=4,
    overwrite=True,
):
    """
    Preprocess an IRIDL monthly wind climatology NetCDF file.

    Parameters
    ----------
    input_path : str or pathlib.Path
        Path to the input NetCDF file.
    output_path : str or pathlib.Path
        Path where the processed NetCDF file will be saved.
    var_name : str, optional
        Name of the wind variable to process. If None, the function expects one data variable.
    target_pressure_level : int or float
        Pressure level to select when a pressure coordinate named P is present.
    compression_level : int
        NetCDF compression level.
    overwrite : bool
        If False and the output file already exists, the function skips processing.
    """
    input_path = Path(input_path)
    output_path = Path(output_path)

    check_input_file(input_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not overwrite:
        print_step(f"Output already exists and overwrite is False. Skipping: {output_path}")
        return xr.open_dataset(output_path)

    print_step(f"Opening input file: {input_path}")
    print_memory_usage("Before opening dataset")

    ds = xr.open_dataset(input_path, decode_times=False)
    print(ds)

    if "T" in ds.coords:
        print_step("Setting T calendar to 360_day and decoding CF time.")
        ds["T"].attrs["calendar"] = "360_day"
        ds = xr.decode_cf(ds, use_cftime=True)
    else:
        print_step("Coordinate T was not found. Time decoding step skipped.")

    if var_name is None:
        data_vars = list(ds.data_vars)
        if len(data_vars) != 1:
            raise ValueError(f"Specify var_name. Variables found: {data_vars}")
        var_name = data_vars[0]

    if var_name not in ds.data_vars:
        raise ValueError(f"Variable {var_name!r} not found. Available variables: {list(ds.data_vars)}")

    print_step(f"Processing variable: {var_name}")
    da = ds[var_name]

    if "P" in da.dims or "P" in da.coords:
        print_step(f"Selecting pressure level P = {target_pressure_level}.")
        da = da.sel(P=target_pressure_level)
        da = da.drop_vars("P", errors="ignore")
    else:
        print_step("Pressure coordinate P was not found. Pressure selection skipped.")

    if "T" in da.dims:
        ntime = da.sizes["T"]
        print_step(f"Detected {ntime} time steps along T.")

        if ntime != 12:
            raise ValueError(f"Expected 12 monthly values, but T has length {ntime}.")

        da = da.assign_coords(month=("T", np.arange(1, 13, dtype=np.int32)))
        da = da.swap_dims({"T": "month"})
        da = da.drop_vars("T", errors="ignore")
    else:
        print_step("Dimension T was not found. Month conversion skipped.")

    rename_dict = {}
    if "Y" in da.dims or "Y" in da.coords:
        rename_dict["Y"] = "latitude"
    if "X" in da.dims or "X" in da.coords:
        rename_dict["X"] = "longitude"

    if rename_dict:
        print_step(f"Renaming coordinates: {rename_dict}")
        da = da.rename(rename_dict)

    required_coords = {"latitude", "longitude"}
    missing_coords = required_coords.difference(set(da.coords).union(set(da.dims)))
    if missing_coords:
        raise ValueError(f"Missing required spatial coordinates after renaming: {missing_coords}")

    print_step("Converting longitude to the -180 / 180 convention.")
    da = da.assign_coords(longitude=((da.longitude + 180) % 360) - 180)
    da = da.sortby("longitude")

    print_step("Sorting latitude from north to south.")
    da = da.sortby("latitude", ascending=False)

    expected_dims = [dim for dim in ["month", "latitude", "longitude"] if dim in da.dims]
    da = da.transpose(*expected_dims)

    out = da.to_dataset(name=var_name)
    out = out[[var_name]]

    if "month" in out.coords:
        out = out.assign_coords(month=out.month.astype("int32"))
    out = out.assign_coords(
        latitude=out.latitude.astype("float32"),
        longitude=out.longitude.astype("float32"),
    )

    out["latitude"].attrs = {
        "standard_name": "latitude",
        "long_name": "latitude",
        "units": "degrees_north",
        "axis": "Y",
    }

    out["longitude"].attrs = {
        "standard_name": "longitude",
        "long_name": "longitude",
        "units": "degrees_east",
        "axis": "X",
    }

    if "month" in out.coords:
        out["month"].attrs = {
            "long_name": "month",
            "units": "1",
        }

    out[var_name].attrs["grid_mapping"] = "crs"

    out["crs"] = xr.DataArray(
        0,
        attrs={
            "grid_mapping_name": "latitude_longitude",
            "epsg_code": "EPSG:4326",
            "spatial_ref": "EPSG:4326",
        },
    )

    encoding = {
        var_name: {
            "zlib": True,
            "complevel": compression_level,
            "_FillValue": np.float32(np.nan),
        }
    }

    print_step(f"Saving processed NetCDF: {output_path}")
    out.to_netcdf(output_path, encoding=encoding)

    ds.close()
    gc.collect()
    print_memory_usage("After saving output")
    print_step(f"Completed preprocessing for variable: {var_name}")

    return out

## Step 5. Optional input preview

Run this cell if you want to inspect the original files before processing.

If the files are large, you can skip this step.

In [12]:
# Uncomment the lines below if you want to inspect the raw datasets.

# u_raw = preview_dataset(INPUT_U_FILE)
# v_raw = preview_dataset(INPUT_V_FILE)

## Step 6. Process the U wind component

This cell creates the processed NetCDF file for the zonal wind component.

In [13]:
u_processed = preprocess_iridl_wind(
    input_path=INPUT_U_FILE,
    output_path=OUTPUT_U_FILE,
    var_name=U_VARIABLE_NAME,
    target_pressure_level=TARGET_PRESSURE_LEVEL,
    compression_level=NETCDF_COMPRESSION_LEVEL,
    overwrite=OVERWRITE_OUTPUTS,
)

u_processed

Input file found: ../data/iri/u_monthly_clim_plev_global.nc
[17:21:22] Opening input file: ../data/iri/u_monthly_clim_plev_global.nc
Before opening dataset: 0.16 GB used by the current Python process
<xarray.Dataset> Size: 9MB
Dimensions:  (Y: 73, X: 144, P: 17, T: 12)
Coordinates:
  * Y        (Y) float32 292B 90.0 87.5 85.0 82.5 ... -82.5 -85.0 -87.5 -90.0
  * X        (X) float32 576B 0.0 2.5 5.0 7.5 10.0 ... 350.0 352.5 355.0 357.5
  * P        (P) int32 68B 1000 925 850 700 600 500 400 ... 100 70 50 30 20 10
  * T        (T) float32 48B 0.5 1.5 2.5 3.5 4.5 5.5 6.5 7.5 8.5 9.5 10.5 11.5
Data variables:
    u        (T, P, Y, X) float32 9MB ...
[17:21:24] Setting T calendar to 360_day and decoding CF time.
[17:21:24] Processing variable: u
[17:21:24] Selecting pressure level P = 1000.
[17:21:24] Detected 12 time steps along T.
[17:21:24] Renaming coordinates: {'Y': 'latitude', 'X': 'longitude'}
[17:21:24] Converting longitude to the -180 / 180 convention.
[17:21:24] Sorting latitude

/tmp/ipykernel_1499308/2990039716.py:46: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.decode_cf(ds, use_cftime=True)


After saving output: 0.28 GB used by the current Python process
[17:21:24] Completed preprocessing for variable: u


<xarray.Dataset> Size: 506kB
Dimensions:    (month: 12, latitude: 73, longitude: 144)
Coordinates:
  * month      (month) int32 48B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude   (latitude) float32 292B 90.0 87.5 85.0 82.5 ... -85.0 -87.5 -90.0
  * longitude  (longitude) float32 576B -180.0 -177.5 -175.0 ... 175.0 177.5
Data variables:
    u          (month, latitude, longitude) float32 505kB ...
    crs        int64 8B 0

## Step 7. Process the V wind component

This cell creates the processed NetCDF file for the meridional wind component.

In [14]:
v_processed = preprocess_iridl_wind(
    input_path=INPUT_V_FILE,
    output_path=OUTPUT_V_FILE,
    var_name=V_VARIABLE_NAME,
    target_pressure_level=TARGET_PRESSURE_LEVEL,
    compression_level=NETCDF_COMPRESSION_LEVEL,
    overwrite=OVERWRITE_OUTPUTS,
)

v_processed

Input file found: ../data/iri/v_monthly_clim_plev_global.nc
[17:21:24] Opening input file: ../data/iri/v_monthly_clim_plev_global.nc
Before opening dataset: 0.28 GB used by the current Python process
<xarray.Dataset> Size: 9MB
Dimensions:  (P: 17, T: 12, Y: 73, X: 144)
Coordinates:
  * P        (P) int32 68B 1000 925 850 700 600 500 400 ... 100 70 50 30 20 10
  * T        (T) float32 48B 0.5 1.5 2.5 3.5 4.5 5.5 6.5 7.5 8.5 9.5 10.5 11.5
  * Y        (Y) float32 292B 90.0 87.5 85.0 82.5 ... -82.5 -85.0 -87.5 -90.0
  * X        (X) float32 576B 0.0 2.5 5.0 7.5 10.0 ... 350.0 352.5 355.0 357.5
Data variables:
    v        (T, P, Y, X) float32 9MB ...
[17:21:24] Setting T calendar to 360_day and decoding CF time.
[17:21:24] Processing variable: v
[17:21:24] Selecting pressure level P = 1000.
[17:21:24] Detected 12 time steps along T.
[17:21:24] Renaming coordinates: {'Y': 'latitude', 'X': 'longitude'}
[17:21:24] Converting longitude to the -180 / 180 convention.
[17:21:24] Sorting latitude

/tmp/ipykernel_1499308/2990039716.py:46: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.decode_cf(ds, use_cftime=True)


<xarray.Dataset> Size: 506kB
Dimensions:    (month: 12, latitude: 73, longitude: 144)
Coordinates:
  * month      (month) int32 48B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude   (latitude) float32 292B 90.0 87.5 85.0 82.5 ... -85.0 -87.5 -90.0
  * longitude  (longitude) float32 576B -180.0 -177.5 -175.0 ... 175.0 177.5
Data variables:
    v          (month, latitude, longitude) float32 505kB ...
    crs        int64 8B 0

## Step 8. Verify the outputs

Run this final check to confirm that the two processed files were created correctly.

In [15]:
print_step("Checking processed output files.")

for output_file in [OUTPUT_U_FILE, OUTPUT_V_FILE]:
    output_file = Path(output_file)
    if not output_file.exists():
        raise FileNotFoundError(f"Expected output file was not created: {output_file}")

    size_mb = output_file.stat().st_size / 1024**2
    print(f"Found output: {output_file} ({size_mb:.2f} MB)")

    with xr.open_dataset(output_file) as ds_check:
        print(ds_check)

print_step("Satellite wind preprocessing completed successfully.")

[17:21:24] Checking processed output files.
Found output: ../data/processed/u_clim/u_iri_processed_global.nc (0.40 MB)
<xarray.Dataset> Size: 506kB
Dimensions:    (month: 12, latitude: 73, longitude: 144)
Coordinates:
  * month      (month) int32 48B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude   (latitude) float32 292B 90.0 87.5 85.0 82.5 ... -85.0 -87.5 -90.0
  * longitude  (longitude) float32 576B -180.0 -177.5 -175.0 ... 175.0 177.5
Data variables:
    u          (month, latitude, longitude) float32 505kB ...
    crs        int64 8B ...
Found output: ../data/processed/v_clim/v_iri_processed_global.nc (0.41 MB)
<xarray.Dataset> Size: 506kB
Dimensions:    (month: 12, latitude: 73, longitude: 144)
Coordinates:
  * month      (month) int32 48B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude   (latitude) float32 292B 90.0 87.5 85.0 82.5 ... -85.0 -87.5 -90.0
  * longitude  (longitude) float32 576B -180.0 -177.5 -175.0 ... 175.0 177.5
Data variables:
    v          (month, latitude, longitude) float3

## Notes for users

If the notebook cannot find the input files, check the paths defined in Step 1.

If the input variable names are different from `u` and `v`, open the optional preview in Step 5 and update `U_VARIABLE_NAME` and `V_VARIABLE_NAME`.

The output files are written in EPSG:4326 and use latitude, longitude and month as standard coordinates.